<a href="https://colab.research.google.com/github/biukaC/DA-Internship-ProgUA/blob/main/task_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [356]:
import pandas as pd
import numpy as np
from datetime import datetime

In [357]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [358]:
task1 = pd.read_csv('/content/drive/MyDrive/DA Internship ProgAcademy/Final version/crm_output_anonymized (1).csv',encoding="utf-8-sig")
task1.head()

,ID,Lead title,Lead sale ₴,Respons. user,Created on,Created by,Modified on,Modified by,Closed at,Tags,...,utm_source.1,referrer,utm_campaign.1,utm_medium.1,utm_content.1,utm_referrer,_ym_uid,_ym_counter,roistat,ga_utm
0,63408688,[NEW] free-data-analytics — /ua/free/data-anal...,0,Vasyl F.,04.06.2026 11:37:01,NaN,04.06.2026 11:39:26,Ivan M.,not closed,NaN,...,meta,NaN,HTTP+|+Data+Analytics+++безкоштовний+курс+|+Si...,cpc,broad+interests+–+27.05,NaN,NaN,NaN,NaN,NaN
1,63408012,[NEW] free-data-analytics — /ua/free/data-anal...,0,Oksana B.,04.06.2026 10:40:14,NaN,04.06.2026 10:45:17,Ivan M.,not closed,NaN,...,meta,NaN,HTTP+|+Data+Analytics+++безкоштовний+курс+|+Si...,cpc,broad+interests+–+27.05,NaN,NaN,NaN,NaN,NaN
2,63407286,form851431729,0,Vasyl F.,04.06.2026 09:42:58,NaN,04.06.2026 10:07:01,Vasyl F.,not closed,"no1, quiz-200125",...,NaN,NaN,NaN,NaN,NaN,https://prog.academy/ua/quiz#apply,NaN,NaN,NaN,NaN
3,63407150,Lead #63407150,0,Vasyl F.,04.06.2026 09:29:40,NaN,04.06.2026 10:06:57,Vasyl F.,not closed,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,63407124,[NEW] free-data-analytics — /ua/free/data-anal...,0,Natalia H.,04.06.2026 09:26:11,NaN,04.06.2026 10:10:57,Ivan M.,not closed,NaN,...,meta,NaN,HTTP+|+Data+Analytics+++безкоштовний+курс+|+Si...,cpc,broad+interests+–+27.05,NaN,NaN,NaN,NaN,NaN


In [359]:
print(task1.shape)

(927, 109)


In [360]:
copy= task1.copy()

In [361]:
date_cols = [
    col for col in copy.columns
    if 'date' in col.lower()
    or 'time' in col.lower()
    or col in ['Created on', 'Modified on', 'Closed at']
]

In [362]:
for col in date_cols:
    copy[col] = pd.to_datetime(copy[col], errors='coerce')

/tmp/ipykernel_909/715457424.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  copy[col] = pd.to_datetime(copy[col], errors='coerce')


In [363]:
for col in date_cols:
    print(f"\n{col}")
    print("NaT count:", copy[col].isna().sum())
    print("NaT %:", round(copy[col].isna().mean() * 100, 2))


Created on
NaT count: 632
NaT %: 68.18

Modified on
NaT count: 448
NaT %: 48.33

Closed at
NaT count: 299
NaT %: 32.25


In [364]:
copy["Created filled"] = copy[["Created on", "Modified on"]].min(axis=1)

In [365]:
created_fill_before = copy["Created on"].notna().mean() * 100
created_fill_after = copy["Created filled"].notna().mean() * 100

print("Created fill BEFORE (%):", round(created_fill_before, 2))
print("Created fill AFTER (%):", round(created_fill_after, 2))

Created fill BEFORE (%): 31.82
Created fill AFTER (%): 60.84


In [366]:
date_cols.append('Created filled')
for col in date_cols:
    print(f"\n{col}")
    print("min:", copy[col].min())
    print("max:", copy[col].max())


Created on
min: 2026-01-05 08:43:30
max: 2026-12-05 22:06:21

Modified on
min: 2026-01-05 10:21:05
max: 2026-12-05 19:05:05

Closed at
min: 2026-01-05 10:21:00
max: 2026-12-05 19:05:00

Created filled
min: 2026-01-05 08:43:30
max: 2026-12-05 22:06:21


In [367]:
now = pd.Timestamp(datetime.now())

for col in date_cols:
    future = copy[copy[col] > now]

    print(f"\n{col}")
    print("future dates:", len(future))


Created on
future dates: 120

Modified on
future dates: 74

Closed at
future dates: 72

Created filled
future dates: 125


In [368]:
now = pd.Timestamp(datetime.now())

mask = (copy[date_cols] > now).any(axis=1)
copy = copy.loc[~mask].copy()

In [369]:
print(copy.shape)

(774, 110)


In [370]:
invalid_modified = copy["Modified on"] < copy["Created filled"]
invalid_closed = copy["Closed at"] < copy["Created filled"]

all_invalid = invalid_modified | invalid_closed

print(f"Найдено строк с ошибочными датами: {all_invalid.sum()}")

copy = copy[~all_invalid]

Найдено строк с ошибочными датами: 116


In [371]:
print(copy.shape)

(658, 110)


In [372]:
utm_cols = [c for c in copy.columns if c.startswith('utm_')]

def norm(x):
    if pd.isna(x):
        return np.nan
    return str(x).strip().lower()

if utm_cols:
    copy[utm_cols] = copy[utm_cols].apply(lambda col: col.map(norm))

groups = {}
for c in utm_cols:
    base = c.split('.')[0]
    groups.setdefault(base, []).append(c)

log_rows = []
clean = pd.DataFrame(index=copy.index)

for base, cols in groups.items():
    merged = copy[cols[0]].copy()

    for col in cols[1:]:
        if col not in copy.columns:
            continue

        a = merged
        b = copy[col]

        both = a.notna() & b.notna()
        conflicts = both & (a != b)

        if conflicts.sum() > 0:
            log_rows.append({
                "field": base,
                "type": "conflict",
                "source_col": cols[0],
                "target_col": col,
                "rows_affected": int(conflicts.sum())
            })

        only_b = b.notna() & a.isna()
        if only_b.sum() > 0:
            log_rows.append({
                "field": base,
                "type": "filled_from_secondary",
                "source_col": col,
                "target_col": cols[0],
                "rows_affected": int(only_b.sum())
            })

        merged = merged.combine_first(b)

    clean[base] = merged

existing_utm_to_drop = [c for c in utm_cols if c in copy.columns]
if existing_utm_to_drop:
    copy = copy.drop(columns=existing_utm_to_drop)

copy = pd.concat([copy, clean], axis=1)

if log_rows:
    utm_report = pd.DataFrame(log_rows)
else:
    utm_report = pd.DataFrame(columns=["field", "type", "source_col", "target_col", "rows_affected"])

print("="*60 + "\n ОТЧЕТ ОБ ОБЪЕДИНЕНИИ И КОНФЛИКТАХ UTM\n" + "="*60)
print(f"Удалено старых UTM-колонок (включая дубликаты): {len(existing_utm_to_drop)}")
print(f"Создано чистых UTM-источников: {len(groups)}")
print(f"Всего зафиксировано событий (конфликты/дополнения): {len(log_rows)}\n")


 ОТЧЕТ ОБ ОБЪЕДИНЕНИИ И КОНФЛИКТАХ UTM
Удалено старых UTM-колонок (включая дубликаты): 11
Создано чистых UTM-источников: 6
Всего зафиксировано событий (конфликты/дополнения): 5



In [373]:
print(copy.shape)

(658, 105)


In [374]:
copy = copy.dropna(axis=1, how='all')
print(copy.shape)

(658, 68)


In [375]:
const_cols_info = {}

for col in copy.columns:
    unique_count = copy[col].nunique(dropna=True)

    if unique_count <= 1:
        val = copy[col].dropna().unique()
        actual_val = val[0] if len(val) > 0 else "Все значения пустые (NaN)"
        const_cols_info[col] = actual_val

print("="*60)
print(" ПРОВЕРКА КОНСТАНТНЫХ КОЛОНОК ДЛЯ УДАЛЕНИЯ ")
print("="*60)
if const_cols_info:
    print(f"Найдено колонок для удаления: {len(const_cols_info)}\n")
    print("Список колонок и их единственные значения:")
    for col, val in const_cols_info.items():
        print(f" • [x] {col} —> Значение: {val}")
else:
    print(" • Константных колонок не найдено. Удаление не требуется.")
print("="*60)

if const_cols_info:
    copy = copy.drop(columns=list(const_cols_info.keys()))
    print(f"Результат: {len(const_cols_info)} колонок успешно удалены из датафрейма [copy].")

 ПРОВЕРКА КОНСТАНТНЫХ КОЛОНОК ДЛЯ УДАЛЕНИЯ 
Найдено колонок для удаления: 5

Список колонок и их единственные значения:
 • [x] Pipeline —> Значение: Продажи
 • [x] Lead's company —> Значение: Company name not specified
 • [x] Пользовательское соглашение —> Значение: No
 • [x] UTM_ID —> Значение: 52635795949090.0
 • [x] FORMNAME —> Значение: Пробный урок
Результат: 5 колонок успешно удалены из датафрейма [copy].


In [376]:
import pandas as pd

threshold = 0.05  # Порог в 5%

# >>> ДОБАВЛЕНО: Создаем словарь для хранения информации о проценте заполненности
low_fill_info = {}

for col in copy.columns:
    fill_rate = copy[col].notna().mean()
    if fill_rate < threshold:
        low_fill_info[col] = fill_rate
# <<<

# >>> ДОБАВЛЕНО: Превращаем ключи словаря в список для удаления
low_fill_cols = list(low_fill_info.keys())
# <<<

# >>> ДОБАВЛЕНО: Печатаем детальный и красивый предварительный отчет
print("="*60)
print(" ПРОВЕРКА МАЛОЗАПОЛНЕННЫХ КОЛОНОК (LOW-FILL) ")
print("="*60)
if low_fill_info:
    print(f"Найдено колонок для удаления (заполнены менее чем на {threshold*100}%): {len(low_fill_info)}\n")
    for col, rate in low_fill_info.items():
        print(f" • [x] {col} —> Заполнена всего на: {rate:.2%}")
else:
    print(" • Все колонки имеют достаточный уровень заполнения данными.")
print("="*60)
# <<<

# Удаляем колонки из нашего рабочего датафрейма copy
if low_fill_cols:
    copy = copy.drop(columns=low_fill_cols)
    print(f"Результат: {len(low_fill_cols)} колонок успешно удалены из датафрейма [copy].")


 ПРОВЕРКА МАЛОЗАПОЛНЕННЫХ КОЛОНОК (LOW-FILL) 
Найдено колонок для удаления (заполнены менее чем на 5.0%): 14

 • [x] Created by —> Заполнена всего на: 1.98%
 • [x] Title —> Заполнена всего на: 4.41%
 • [x] TRAF_SRC —> Заполнена всего на: 1.67%
 • [x] TRAF_TYPE —> Заполнена всего на: 1.67%
 • [x] ADV_CAMP —> Заполнена всего на: 1.67%
 • [x] KEYWORD —> Заполнена всего на: 1.67%
 • [x] TRAF_CONT —> Заполнена всего на: 1.67%
 • [x] GOOGLE_ID —> Заполнена всего на: 3.34%
 • [x] Комментарий —> Заполнена всего на: 2.58%
 • [x] COURSE —> Заполнена всего на: 4.71%
 • [x] UTM_ADGROUP —> Заполнена всего на: 1.82%
 • [x] GOOGLE_ID.1 —> Заполнена всего на: 0.76%
 • [x] fbclid —> Заполнена всего на: 0.61%
 • [x] gclientid —> Заполнена всего на: 4.10%
Результат: 14 колонок успешно удалены из датафрейма [copy].


In [377]:
duplicate_cols = []

cols = copy.columns

for i in range(len(cols)):
    for j in range(i + 1, len(cols)):

        c1 = cols[i]
        c2 = cols[j]

        if copy[c1].fillna('__NA__').equals(copy[c2].fillna('__NA__')):
            duplicate_cols.append(c2)

df_clean = copy.drop(columns=duplicate_cols)

print("Удалено дублей:", len(set(duplicate_cols)))

Удалено дублей: 0


In [378]:
text_cols = copy.select_dtypes(include='object').columns

low_info_cols = []

for col in text_cols:
    if copy[col].nunique(dropna=True) > 0:
        entropy = copy[col].nunique() / copy[col].notna().sum()

        if entropy > 0.8:
            low_info_cols.append(col)

print(low_info_cols)

['Note', 'Note 3', 'Note 4', 'Note 5', 'Work phone', 'Mobile phone', 'Other phone', 'Work email', 'Personal email', 'Other email', 'Telegram', 'REFERER', 'TRANID', 'COOKIES', 'gclid', 'referrer']


In [379]:
copy["phone"] = (
    copy["Work phone"]
    .combine_first(copy["Mobile phone"])
    .combine_first(copy["Other phone"])
)

copy = copy.drop(columns=[
    "Work phone",
    "Mobile phone",
    "Other phone"
])

copy["email"] = (
    copy["Work email"]
    .combine_first(copy["Personal email"])
    .combine_first(copy["Other email"])
)

copy = copy.drop(columns=[
    "Work email",
    "Personal email",
    "Other email"
])

copy["note_full"] = (
    copy["Note"].fillna("") + " | " +
    copy["Note 3"].fillna("") + " | " +
    copy["Note 4"].fillna("") + " | " +
    copy["Note 5"].fillna("")
).str.strip(" |")

copy = copy.drop(columns=[
    "Note", "Note 3", "Note 4", "Note 5"
])

In [380]:
copy.head()

,ID,Lead title,Lead sale ₴,Respons. user,Created on,Modified on,Modified by,Closed at,Tags,Note 2,...,Created filled,utm_source,utm_medium,utm_campaign,utm_term,utm_content,utm_referrer,phone,email,note_full
0,63408688,[NEW] free-data-analytics — /ua/free/data-anal...,0,Vasyl F.,2026-04-06 11:37:01,2026-04-06 11:39:26,Ivan M.,NaT,NaN,NaN,...,2026-04-06 11:37:01,meta,cpc,http+|+data+analytics+++безкоштовний+курс+|+si...,статика+6,broad+interests+–+27.05,NaN,'+380732768040,gpossk469@yahoo.com,'=== First Touch ===\nutm_source: meta\nutm_me...
1,63408012,[NEW] free-data-analytics — /ua/free/data-anal...,0,Oksana B.,2026-04-06 10:40:14,2026-04-06 10:45:17,Ivan M.,NaT,NaN,NaN,...,2026-04-06 10:40:14,meta,cpc,http+|+data+analytics+++безкоштовний+курс+|+si...,статика+6,broad+interests+–+27.05,NaN,'+380994950580,rgdxg618@gmail.com,'=== First Touch ===\nutm_source: meta\nutm_me...
2,63407286,form851431729,0,Vasyl F.,2026-04-06 09:42:58,2026-04-06 10:07:01,Vasyl F.,NaT,"no1, quiz-200125",NaN,...,2026-04-06 09:42:58,NaN,NaN,NaN,NaN,NaN,https://prog.academy/ua/quiz#apply,'+380682149696,kvhfupi462@meta.ua,FE: 10\nDA: 2\nQA: 2
3,63407150,Lead #63407150,0,Vasyl F.,2026-04-06 09:29:40,2026-04-06 10:06:57,Vasyl F.,NaT,NaN,NaN,...,2026-04-06 09:29:40,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,
4,63407124,[NEW] free-data-analytics — /ua/free/data-anal...,0,Natalia H.,2026-04-06 09:26:11,2026-04-06 10:10:57,Ivan M.,NaT,NaN,NaN,...,2026-04-06 09:26:11,meta,cpc,http+|+data+analytics+++безкоштовний+курс+|+si...,статика+6,broad+interests+–+27.05,NaN,'+380630563876,udbzpt476@meta.ua,'=== First Touch ===\nutm_source: meta\nutm_me...


In [381]:
print(copy.shape)
copy.columns.tolist()

(658, 42)


['ID',
 'Lead title',
 'Lead sale ₴',
 'Respons. user',
 'Created on',
 'Modified on',
 'Modified by',
 'Closed at',
 'Tags',
 'Note 2',
 'Lead stage',
 'Fullname',
 'Contact`s company',
 'Contact respons. user',
 'Telegram',
 'Оплата',
 'Курс',
 'Lead scoring',
 'Название пакета',
 'Язык',
 'REFERER',
 'FORMID',
 'TRANID',
 'COOKIES',
 'Вопрос 1',
 'Вопрос 2',
 'Вопрос 3',
 'Вопрос 4',
 'Вопрос 5',
 'gclid',
 'from',
 'referrer',
 'Created filled',
 'utm_source',
 'utm_medium',
 'utm_campaign',
 'utm_term',
 'utm_content',
 'utm_referrer',
 'phone',
 'email',
 'note_full']

In [382]:
copy.dtypes

,0
ID,int64
Lead title,object
Lead sale ₴,int64
Respons. user,object
Created on,datetime64[ns]
Modified on,datetime64[ns]
Modified by,object
Closed at,datetime64[ns]
Tags,object
Note 2,object


In [383]:
copy.isnull().sum().sort_values(ascending=False)

,0
Название пакета,625
FORMID,623
REFERER,623
from,623
TRANID,623
Язык,623
COOKIES,623
referrer,617
Вопрос 5,611
gclid,610


In [384]:
copy.to_csv('/content/drive/MyDrive/DA Internship ProgAcademy/Final version/copy1.csv', index=False, encoding='utf-8-sig')